In [ ]:
pip install selenium

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=options)
driver.get("https://www.google.com")
print("✅ Funcionando:", driver.title)
driver.quit()

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

options = Options()
# Testa SEM headless — janela visível tem menos chance de ser bloqueada
# options.add_argument("--headless=new")  # deixa comentado por enquanto
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-extensions")
options.add_argument("--disable-popup-blocking")
options.add_argument("--profile-directory=Default")
options.add_argument("--disable-plugins-discovery")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=options)

# Oculta propriedades de automação
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {"source": """
    Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
    Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3]});
    Object.defineProperty(navigator, 'languages', {get: () => ['pt-BR', 'pt', 'en-US']});
    window.chrome = { runtime: {} };
"""})

# Acessa o Google antes para simular comportamento humano
driver.get("https://www.google.com.br")
time.sleep(2)

URL = (
    "https://www.tripadvisor.com.br/Attraction_Review-g680306-d2401618"
    "-Reviews-Praia_das_Laranjeiras-Balneario_Camboriu_State_of_Santa_Catarina.html"
)
driver.get(URL)
time.sleep(6)  # aguarda mais tempo

# Simula scroll para parecer humano
driver.execute_script("window.scrollTo(0, 300);")
time.sleep(1)
driver.execute_script("window.scrollTo(0, 600);")
time.sleep(1)

print("📄 Título da página:", driver.title)

soup = BeautifulSoup(driver.page_source, "html.parser")

def extract_reviews(soup: BeautifulSoup) -> list:

    reviews = []

    card_wrapper = soup.find(class_="mSOQy emJoM")
    cards = card_wrapper.find_all(class_="_c")

    if len(cards) > 0:
        print("-" * 40)

    for card in cards:
        user = card.find(class_="QIHsu Zb").a.text.strip()
        rating = card.find(class_="MyMKp u Q1").title.text.split()[0]
        comment = card.find(class_="JguWG").text.strip()
        comment_title = card.find(class_="biGQs _P SewaP qWPrE ncFvv ezezH").text.strip()

        local = card.find(class_="biGQs _P navcl").span.text
        if local[0] >= "0" and local[0] <= "9":
            local = None

        date = card.find(class_="BNelO").div.text[9:]
        
        reviews.append({
            "user": user,
            "rating": rating,
            "comment_title": comment_title,
            "comment": comment,
            "local": local,
            "date": date
        })

        # print(f"User: {user}")
        # print(f"Rating: {rating}")
        # print(f"Comment Title: {comment_title}")
        # print(f"Comment: {comment}")
        # print(f"Local: {local}")
        # print(f"Date: {date}")
        # print("-" * 40)

    return reviews

reviews = extract_reviews(soup)
print(f"Total de avaliações extraídas: {len(reviews)}")